# Hugging face -> Torch -> Edge IR -> CoreML backend [fp 16] -> .pte

In [26]:
import torch
import coremltools as ct
from transformers import AutoModelForCausalLM, AutoTokenizer
from executorch import exir
from executorch.exir import to_edge, EdgeCompileConfig
from executorch.backends.apple.coreml.partition import CoreMLPartitioner
from executorch.backends.apple.coreml.quantizer import CoreMLQuantizer
from torch.ao.quantization.quantize_pt2e import prepare_pt2e, convert_pt2e
from torch.export import export, export_for_training

In [27]:
import os, pprint

# Reliable HF download behavior in notebooks (set inside kernel)
os.environ["HF_HUB_READ_TIMEOUT"] = "120"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"   # enable faster transfer path
os.environ["HF_HUB_DOWNLOAD_RETRY"] = "10"
os.environ["EXECUTORCH_LOG_LEVEL"] = "debug"
os.environ["TRANSFORMERS_VERBOSITY"] = "info"
os.environ["HF_HUB_VERBOSITY"] = "debug"

# Quick check of envvars from inside the kernel
vars_to_check = [
    "HF_HUB_READ_TIMEOUT",
    "HF_HUB_ENABLE_HF_TRANSFER",
    "HF_HUB_DOWNLOAD_RETRY",
    "EXECUTORCH_LOG_LEVEL",
    "TRANSFORMERS_VERBOSITY",
    "HF_HUB_VERBOSITY",
]
print("Environment (checked from Python):")
pprint.pprint({v: os.environ.get(v) for v in vars_to_check})


Environment (checked from Python):
{'EXECUTORCH_LOG_LEVEL': 'debug',
 'HF_HUB_DOWNLOAD_RETRY': '10',
 'HF_HUB_ENABLE_HF_TRANSFER': '1',
 'HF_HUB_READ_TIMEOUT': '120',
 'HF_HUB_VERBOSITY': 'debug',
 'TRANSFORMERS_VERBOSITY': 'info'}


In [28]:
from transformers import logging
from huggingface_hub.utils import logging as hf_logging

# Transformers verbosity (info -> debug if you want even more)
logging.set_verbosity_info()      # or set_verbosity_debug() if you like noise
logging.enable_default_handler()
logging.enable_explicit_format()

# HF hub HTTP-level logs
hf_logging.set_verbosity_debug()


In [29]:
class TraceableWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        # Force disable cache to prevent DynamicCache errors
        self.model.config.use_cache = False 

    def forward(self, input_ids):
        # We only return logits. 
        # use_cache=False is redundant if config is set, but safe.
        output = self.model(input_ids, use_cache=False)
        return output.logits

In [30]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import os
from safetensors.torch import save_file

MODEL_ID = "Qwen/Qwen3-1.7B"   # change if you have a local mirror
OUT_DIR = "qwen3_fp16"

os.makedirs(OUT_DIR, exist_ok=True)

print("Loading model (this may take a while; verbose logs above will show progress)...")
raw_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,            # use dtype= (not torch_dtype) in newer transformers
    device_map="cpu",
    trust_remote_code=True,         # required for Qwen-3
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# Save tokenizer and HF config (these calls will write config.json etc)
#tokenizer.save_pretrained(OUT_DIR)
model = TraceableWrapper(raw_model)
#raw_model.save_pretrained(OUT_DIR) 
model.eval()

Request 290caeff-13c3-4dd4-a897-d3f105cbad3c: HEAD https://huggingface.co/Qwen/Qwen3-1.7B/resolve/main/config.json (authenticated: True)
DEBUG:huggingface_hub.utils._http:Request 290caeff-13c3-4dd4-a897-d3f105cbad3c: HEAD https://huggingface.co/Qwen/Qwen3-1.7B/resolve/main/config.json (authenticated: True)


Loading model (this may take a while; verbose logs above will show progress)...


Request a78a132a-7ce0-425c-ab29-996806d8c80a: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-1.7B/70d244cc86ccca08cf5af4e1e306ecf908b1ad5e/config.json (authenticated: True)
DEBUG:huggingface_hub.utils._http:Request a78a132a-7ce0-425c-ab29-996806d8c80a: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen3-1.7B/70d244cc86ccca08cf5af4e1e306ecf908b1ad5e/config.json (authenticated: True)
[INFO|configuration_utils.py:765] 2026-02-09 12:01:38,459 >> loading configuration file config.json from cache at /Users/naren_work/.cache/huggingface/hub/models--Qwen--Qwen3-1.7B/snapshots/70d244cc86ccca08cf5af4e1e306ecf908b1ad5e/config.json
[INFO|configuration_utils.py:839] 2026-02-09 12:01:38,466 >> Model config Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "float16",
  "eos_token_id": 151645,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initiali

TraceableWrapper(
  (model): Qwen3ForCausalLM(
    (model): Qwen3Model(
      (embed_tokens): Embedding(151936, 2048)
      (layers): ModuleList(
        (0-27): 28 x Qwen3DecoderLayer(
          (self_attn): Qwen3Attention(
            (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
            (k_proj): Linear(in_features=2048, out_features=1024, bias=False)
            (v_proj): Linear(in_features=2048, out_features=1024, bias=False)
            (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
            (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
            (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          )
          (mlp): Qwen3MLP(
            (gate_proj): Linear(in_features=2048, out_features=6144, bias=False)
            (up_proj): Linear(in_features=2048, out_features=6144, bias=False)
            (down_proj): Linear(in_features=6144, out_features=2048, bias=False)
            (act_fn): SiLU()
          )
          (input_layernorm): Qwen3

In [31]:
import torch
import coremltools as ct
from torch.export import export_for_training, export
from executorch.backends.apple.coreml.partition import CoreMLPartitioner
from executorch.exir import to_edge

In [32]:
# --- 0. example input (adjust length to your model's expected context length) ---
example_input = (torch.randint(0, tokenizer.vocab_size, (1, 1024), dtype = torch.long,),)
model.eval()
exported_program = export_for_training(model, example_input)
print(exported_program)

/var/folders/ct/rv3z52fx5298nlp84rc1vf1c0000gq/T/ipykernel_25340/2025343562.py:4: FutureWarning: `torch.export.export_for_training` is deprecated and will be removed in PyTorch 2.10. Please use `torch.export.export` instead, which is functionally equivalent.
  exported_program = export_for_training(model, example_input)


ExportedProgram:
    class GraphModule(torch.nn.Module):
        def forward(self, p_model_model_embed_tokens_weight: "f16[151936, 2048]", p_model_model_layers_0_self_attn_q_proj_weight: "f16[2048, 2048]", p_model_model_layers_0_self_attn_k_proj_weight: "f16[1024, 2048]", p_model_model_layers_0_self_attn_v_proj_weight: "f16[1024, 2048]", p_model_model_layers_0_self_attn_o_proj_weight: "f16[2048, 2048]", p_model_model_layers_0_self_attn_q_norm_weight: "f16[128]", p_model_model_layers_0_self_attn_k_norm_weight: "f16[128]", p_model_model_layers_0_mlp_gate_proj_weight: "f16[6144, 2048]", p_model_model_layers_0_mlp_up_proj_weight: "f16[6144, 2048]", p_model_model_layers_0_mlp_down_proj_weight: "f16[2048, 6144]", p_model_model_layers_0_input_layernorm_weight: "f16[2048]", p_model_model_layers_0_post_attention_layernorm_weight: "f16[2048]", p_model_model_layers_1_self_attn_q_proj_weight: "f16[2048, 2048]", p_model_model_layers_1_self_attn_k_proj_weight: "f16[1024, 2048]", p_model_model_layers

Edge IR (Edge Intermediate Representation) is ExecuTorch’s internal, device-safe graph format—the form your model must be in before it can be lowered to CoreML, XNNPACK, Vulkan, Metal, etc.


In [33]:
from executorch.exir import to_edge

edge_program = to_edge(exported_program)
print(edge_program)

Exception ignored in: <function WeakIdKeyDictionary.__init__.<locals>.remove at 0x3d35dbe20>
Traceback (most recent call last):
  File "/Users/naren_work/Code/et_coreml/lib/python3.11/site-packages/torch/utils/weak.py", line 159, in remove
    def remove(k, selfref=ref(self)):

KeyboardInterrupt: 


Lower Edge IR into specific backend. In this case CoreML

In [ ]:
from executorch.backends.apple.coreml.compiler import CoreMLBackend

compile_specs = CoreMLBackend.generate_compile_specs(
        compute_unit=ct.ComputeUnit.ALL,
        compute_precision=ct.precision.FLOAT16,
    )

str(compile_specs)

"[CompileSpec(key='compute_units', value=b'all'), CompileSpec(key='min_deployment_target', value=b''), CompileSpec(key='model_compute_precision', value=b'float16'), CompileSpec(key='model_type', value=b'model')]"

In [ ]:
partitioner = CoreMLPartitioner(
    compile_specs=compile_specs
)

edge_program = edge_program.to_backend(partitioner)
str(edge_program)

Running MIL backend_mlprogram pipeline:   0%|          | 0/12 [00:00<?, ? passes/s]WARNING:coremltools:
Output 'aten_view_copy_default_739' is of dtype fp16. The Core ML runtime does not support outputs with this dtype (supported dtypes are: {<class 'coremltools.converters.mil.mil.types.type_double.make_float.<locals>.double'>, <class 'coremltools.converters.mil.mil.types.type_int.make_int.<locals>.int'>}). This output will changed to fp32 by adding a cast op at the end of the program.


Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 30.65 passes/s]


'<executorch.exir.program._program.EdgeProgramManager object at 0x3d0cbf3d0>'

Final executorch runnable program

In [ ]:
executorch_program = edge_program.to_executorch()

Final Executorch model .pte file

In [ ]:
with open("./models/qwen3_1_7b_coreml_fp16_1024.pte", "wb") as file:
    file.write(executorch_program.buffer)

# Misc code

In [ ]:


quantization_config = ct.optimize.torch.quantization.LinearQuantizerConfig.from_dict({
   # "global_config": {
    #    "quantization_scheme": ct.optimize.torch.quantization.QuantizationScheme.symmetric,
       # "weight_dtype": torch.qint8,
    #    "weight_per_channel": True,
   # },
    "global_config": {
       # torch.nn.Linear: {
           # "activation_dtype": "None",
            "weight_dtype": torch.qint8,
            "quantization_scheme": ct.optimize.torch.quantization.QuantizationScheme.symmetric,
            "weight_per_channel": True,
       # }
    }
})

# --- 3. instantiate the CoreMLQuantizer with the coremltools config ---
coreml_quantizer = CoreMLQuantizer(quantization_config)  

# --- 4. PT2E prepare, calibrate, convert ---
prepared = prepare_pt2e(training_gm, quantizer=coreml_quantizer)

# calibriation: run a few representative batches through the prepared graph:
with torch.no_grad():
    for _ in range(8):                          # run several representative batches
        prepared(*example_input)



/var/folders/ct/rv3z52fx5298nlp84rc1vf1c0000gq/T/ipykernel_3928/1812198102.py:14: FutureWarning: `torch.export.export_for_training` is deprecated and will be removed in PyTorch 2.10. Please use `torch.export.export` instead, which is functionally equivalent.
  training_gm = export_for_training(model, example_input).module()
/var/folders/ct/rv3z52fx5298nlp84rc1vf1c0000gq/T/ipykernel_3928/1812198102.py:37: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quan

NotImplementedError: "fake_quantize_tensor_cachemask_kernel_type_handling" not implemented for 'Long'

In [ ]:
quantized_gm = convert_pt2e(prepared)

In [ ]:
# --- 3. Lower to Edge IR ---
edge_program = to_edge(exported_program)

print("Partitioning for CoreML with 4-bit weight quantization...")

compile_specs = [CoreMLBackend.generate_compile_specs(
    compute_unit=ct.ComputeUnit.ALL,
    compute_precision=ct.precision.FLOAT16,
)]

partitioner = CoreMLPartitioner(compile_specs=compile_specs)
edge_program = edge_program.to_backend(partitioner)

Partitioning for CoreML with 4-bit weight quantization...


TypeError: CoreMLBackend.generate_compile_specs() got an unexpected keyword argument 'weight_quantization'

In [ ]:
# --- 5. EXPORT TO EXECUTORCH ---
print("Exporting final program...")
# We use export() now, not export_for_training
final_export = export(quantized_model, example_input)
edge_program = to_edge(final_export)

[]